In [ ]:
import json, numpy as np, tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models
from sklearn.utils.class_weight import compute_class_weight

IMG = (224, 224)
BATCH = 32
EPOCHS = 8
tf.keras.utils.set_random_seed(42)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow 2.21.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
!pip install -q importlib_resources

## 1. Download PlantVillage and split it 80 / 10 / 10

In [ ]:
splits = ["train[:80%]", "train[80%:90%]", "train[90%:]"]
(train_raw, val_raw, test_raw), info = tfds.load(
    "plant_village", split=splits, as_supervised=True, with_info=True)
class_names = info.features["label"].names
num_classes = len(class_names)
print(num_classes, "classes, e.g.", class_names[:3])

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/plant_village/incomplete.HBRPOI_1.0.2/plant_village-train.tfrecord-[0-9][0…

Dataset plant_village downloaded and prepared to /root/tensorflow_datasets/plant_village/1.0.2. Subsequent calls will reuse this data.
38 classes, e.g. ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust']


## 2. Class weights (fixes the class imbalance without copying images)

In [ ]:
labels = np.concatenate([y.numpy() for y in train_raw.map(lambda x, y: y).batch(4096)])
counts = np.bincount(labels, minlength=num_classes)
print(f"Imbalance: biggest class has {counts.max() / counts.min():.0f}x more images than the smallest")
weights = compute_class_weight("balanced", classes=np.arange(num_classes), y=labels)
class_weight = dict(enumerate(weights))

Imbalance: biggest class has 36x more images than the smallest


## 3. Prepare images and augmentation (training images only)

In [ ]:
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
])

def prep(x, y):
    return tf.image.resize(x, IMG), y          # keeps pixel values 0-255; the model rescales them itself

train_ds = (train_raw.map(prep, tf.data.AUTOTUNE).shuffle(2000, seed=42).batch(BATCH)
            .map(lambda x, y: (augment(x, training=True), y), tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE))
val_ds = val_raw.map(prep, tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)
test_ds = test_raw.map(prep, tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

## 4. Build and train the model (frozen MobileNetV2 + small classifier head)

In [ ]:
base = tf.keras.applications.MobileNetV2(input_shape=IMG + (3,), include_top=False, weights="imagenet")
base.trainable = False

inputs = layers.Input(shape=IMG + (3,))
x = layers.Rescaling(1 / 127.5, offset=-1)(inputs)     # MobileNetV2 expects values in [-1, 1]
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)
model = models.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, class_weight=class_weight)

Epoch 1/8
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 513s 359ms/step - accuracy: 0.7330 - loss: 0.9292 - val_accuracy: 0.8783 - val_loss: 0.3903
Epoch 2/8
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 488s 358ms/step - accuracy: 0.8458 - loss: 0.4854 - val_accuracy: 0.9098 - val_loss: 0.2847
Epoch 3/8
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 464s 340ms/step - accuracy: 0.8637 - loss: 0.4160 - val_accuracy: 0.9072 - val_loss: 0.2838
Epoch 4/8
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 514s 349ms/step - accuracy: 0.8746 - loss: 0.3818 - val_accuracy: 0.9046 - val_loss: 0.2838
Epoch 5/8
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 511s 355ms/step - accuracy: 0.8761 - loss: 0.3742 - val_accuracy: 0.9168 - val_loss: 0.2443
Epoch 6/8
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 473s 347ms/step - accuracy: 0.8853 - loss: 0.3488 - val_accuracy: 0.9144 - val_loss: 0.2591
Epoch 7/8
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 481s 353ms/step - accuracy: 0.8871 - loss: 0.3433 - val_accuracy: 0.9295 - val_loss: 0.2170
Epoch 8/8
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 478s 350ms/step - accuracy: 

## 5. Test on images the model has never seen

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"\nTEST ACCURACY: {test_acc * 100:.2f}%   <- write THIS number in your resume and README")

170/170 ━━━━━━━━━━━━━━━━━━━━ 23s 134ms/step - accuracy: 0.9129 - loss: 0.2606

TEST ACCURACY: 91.29%   <- write THIS number in your resume and README


## 6. Convert to TensorFlow Lite (for the phone) and check it still works

In [ ]:
model.save("plantcare_model.keras")
with open("labels.json", "w") as f:
    json.dump(class_names, f, indent=2)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
open("plantcare.tflite", "wb").write(converter.convert())

interp = tf.lite.Interpreter(model_path="plantcare.tflite")
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
right = total = 0
for x, y in test_raw.map(prep).take(300):
    interp.set_tensor(inp["index"], np.expand_dims(x.numpy().astype("float32"), 0))
    interp.invoke()
    right += int(np.argmax(interp.get_tensor(out["index"])[0]) == y.numpy())
    total += 1
import os
print(f"TFLite size: {os.path.getsize('plantcare.tflite') / 1e6:.1f} MB | accuracy on {total} test images: {right / total * 100:.1f}%")

Saved artifact at '/tmp/tmpueu7cltr'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_319')
Output Type:
  TensorSpec(shape=(None, 38), dtype=tf.float32, name=None)
Captures:
  136710360148560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707230193552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707230195472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707230195856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707230193360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707230196240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707230197200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707230197008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707230197392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707230196048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707230

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


TFLite size: 2.7 MB | accuracy on 300 test images: 92.0%


## 7. Download your files

In [ ]:
from google.colab import files
files.download("plantcare.tflite")
files.download("labels.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>